# 03.2 — Treinamento dos Modelos
**Depende de:** 03.1 executado antes (artefatos em out/artifacts/)
**Salva em:** out/artifacts/ e out/03.2_treinamento/

In [1]:
from __future__ import annotations

import json
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.4f}".format)

# ── Idioma / Language ─────────────────────────────────────────────────────────
LANG = "pt"  # "pt" | "en"  ← única linha a alterar para mudar idioma
# (este arquivo não gera figuras — flag mantida por consistência com os demais)

# Paths
ROOT    = Path("..").resolve()
DATA_DIR = ROOT / "data"
OUT_DIR  = ROOT / "out" / "03.2_treinamento"
PLOT_DIR = OUT_DIR / "plots"
ART_DIR  = ROOT / "out" / "artifacts"   # shared state between sub-files

OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR.mkdir(parents=True, exist_ok=True)

## Carregar Artefatos de 03.1

In [2]:
with open(ART_DIR / "splits.pkl", "rb") as f:
    _splits = pickle.load(f)

df_train      = _splits["df_train"]
df_valid      = _splits["df_valid"]
df_test       = _splits["df_test"]
y_train       = _splits["y_train"]
y_valid       = _splits["y_valid"]
y_test        = _splits["y_test"]
X_train_t     = _splits["X_train_t"]
X_valid_t     = _splits["X_valid_t"]
X_test_t      = _splits["X_test_t"]
X_train       = _splits["X_train"]
X_valid       = _splits["X_valid"]
X_test        = _splits["X_test"]

with open(ART_DIR / "features.json", "r", encoding="utf-8") as f:
    _feat = json.load(f)

FEATURES                = _feat["FEATURES"]
FEATURES_ADMISSAO       = _feat["FEATURES_ADMISSAO"]
FEATURES_POS_ATENDIMENTO= _feat["FEATURES_POS_ATENDIMENTO"]
FEATURES_ALL            = _feat["FEATURES_ALL"]
TARGET_PRINCIPAL        = _feat["TARGET_PRINCIPAL"]
THRESHOLD               = _feat["THRESHOLD"]
TARGETS                 = _feat["TARGETS"]

with open(ART_DIR / "preprocessor.pkl", "rb") as f:
    _preproc_d = pickle.load(f)
preproc = _preproc_d["preproc"]

with open(ART_DIR / "feat_names.json", "r", encoding="utf-8") as f:
    feat_names = json.load(f)

print(f"Splits carregados: treino={len(df_train):,}  valid={len(df_valid):,}  teste={len(df_test):,}")
print(f"FEATURES: {len(FEATURES)}  TARGET: {TARGET_PRINCIPAL}")

CORES = ["steelblue", "darkorange", "seagreen", "firebrick",
         "mediumpurple", "saddlebrown", "deeppink", "teal"]

## 6. Funções de Avaliação — Métricas Estendidas

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    roc_curve, precision_recall_curve, matthews_corrcoef,
)

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("XGBoost não instalado: pip install xgboost")

try:
    from lightgbm import LGBMClassifier
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print("LightGBM não instalado: pip install lightgbm")

try:
    from catboost import CatBoostClassifier
    HAS_CAT = True
except ImportError:
    HAS_CAT = False
    print("CatBoost não instalado: pip install catboost")

from sklearn.tree import DecisionTreeClassifier


# _Dummy definido no escopo do módulo para que o pickle seja serializado/desserializado
# corretamente em qualquer notebook que carregue modelos_fit.pkl.
class _Dummy:
    """Baseline de prevalência — prediz sempre a taxa marginal do treino."""
    def __init__(self, prev=0.5):
        self.prev = prev
    def predict_proba(self, X):
        n = len(X) if hasattr(X, "__len__") else X.shape[0]
        return np.column_stack([1 - self.prev * np.ones(n),
                                       self.prev * np.ones(n)])


def ece_score(y_true, y_prob, n_bins=10):
    """Expected Calibration Error."""
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (y_prob >= lo) & (y_prob < hi)
        if mask.sum() == 0:
            continue
        acc  = y_true[mask].mean()
        conf = y_prob[mask].mean()
        ece += mask.sum() / len(y_true) * abs(acc - conf)
    return ece


def youden_threshold(y_true, y_prob):
    """Threshold ótimo via Youden's J (maximiza sens+spec-1)."""
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    j = tpr - fpr
    return float(thresholds[np.argmax(j)])


def avaliar(y_true, y_prob, threshold=0.30, n_boot=500, threshold_opt=None):
    """Métricas + IC95% via bootstrap. Inclui NPV, MCC, F1, e métricas @Youden."""
    y_true = np.array(y_true)
    y_prob = np.array(y_prob)
    y_pred = (y_prob >= threshold).astype(int)

    if threshold_opt is None:
        try:
            threshold_opt = youden_threshold(y_true, y_prob)
        except Exception:
            threshold_opt = threshold
    y_pred_opt = (y_prob >= threshold_opt).astype(int)

    def _metrics(yt, yp_prob, yp_bin, yp_bin_opt=None):
        if yt.sum() == 0 or yt.sum() == len(yt):
            return {}
        tp = ((yt == 1) & (yp_bin == 1)).sum()
        tn = ((yt == 0) & (yp_bin == 0)).sum()
        fp = ((yt == 0) & (yp_bin == 1)).sum()
        fn = ((yt == 1) & (yp_bin == 0)).sum()
        sens = tp / (tp + fn + 1e-9)
        spec = tn / (tn + fp + 1e-9)
        ppv  = tp / (tp + fp + 1e-9)
        npv  = tn / (tn + fn + 1e-9)
        f1   = 2 * tp / (2 * tp + fp + fn + 1e-9)
        try:
            mcc = matthews_corrcoef(yt, yp_bin)
        except Exception:
            mcc = np.nan

        result = dict(
            auroc = roc_auc_score(yt, yp_prob),
            auprc = average_precision_score(yt, yp_prob),
            brier = brier_score_loss(yt, yp_prob),
            ece   = ece_score(yt, yp_prob),
            sens  = sens,
            spec  = spec,
            ppv   = ppv,
            npv   = npv,
            f1    = f1,
            mcc   = mcc,
        )

        # Métricas @Youden threshold
        if yp_bin_opt is not None:
            tp_o = ((yt == 1) & (yp_bin_opt == 1)).sum()
            tn_o = ((yt == 0) & (yp_bin_opt == 0)).sum()
            fp_o = ((yt == 0) & (yp_bin_opt == 1)).sum()
            fn_o = ((yt == 1) & (yp_bin_opt == 0)).sum()
            result["sens_youden"] = tp_o / (tp_o + fn_o + 1e-9)
            result["spec_youden"] = tn_o / (tn_o + fp_o + 1e-9)
            result["f1_youden"]   = 2 * tp_o / (2 * tp_o + fp_o + fn_o + 1e-9)

        return result

    base = _metrics(y_true, y_prob, y_pred, y_pred_opt)
    if not base:
        return base

    boots = []
    rng = np.random.default_rng(42)
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_true), len(y_true))
        yb  = y_true[idx]; pb = y_prob[idx]
        if yb.sum() == 0 or yb.sum() == len(yb):
            continue
        try:
            thr_b = youden_threshold(yb, pb)
        except Exception:
            thr_b = threshold_opt
        boots.append(_metrics(yb, pb, (pb >= threshold).astype(int),
                               (pb >= thr_b).astype(int)))

    out = {}
    for k, v in base.items():
        boot_vals = [b[k] for b in boots if k in b and not np.isnan(b[k])]
        lo = np.percentile(boot_vals, 2.5)  if boot_vals else np.nan
        hi = np.percentile(boot_vals, 97.5) if boot_vals else np.nan
        out[k]         = round(v, 4) if not np.isnan(v) else np.nan
        out[f"{k}_lo"] = round(lo, 4) if not np.isnan(lo) else np.nan
        out[f"{k}_hi"] = round(hi, 4) if not np.isnan(hi) else np.nan

    out["threshold_opt"] = round(threshold_opt, 4)
    return out


def nri_idi(y_true, prob_ref, prob_new):
    """
    NRI (Net Reclassification Improvement) e IDI (Integrated Discrimination Improvement).
    NRI contínuo: mean(sign(prob_new - prob_ref)[y==1]) + mean(sign(prob_ref - prob_new)[y==0])
    IDI: [mean(prob_new[y==1]) - mean(prob_new[y==0])] - [mean(prob_ref[y==1]) - mean(prob_ref[y==0])]
    """
    y_true   = np.array(y_true)
    prob_ref = np.array(prob_ref)
    prob_new = np.array(prob_new)

    pos_mask = (y_true == 1)
    neg_mask = (y_true == 0)

    diff = prob_new - prob_ref

    nri_events   = np.mean(np.sign(diff[pos_mask]))  if pos_mask.sum() > 0 else np.nan
    nri_noevents = np.mean(np.sign(-diff[neg_mask]))  if neg_mask.sum() > 0 else np.nan
    nri = nri_events + nri_noevents

    idi_events   = prob_new[pos_mask].mean() - prob_ref[pos_mask].mean() if pos_mask.sum() > 0 else np.nan
    idi_noevents = prob_new[neg_mask].mean() - prob_ref[neg_mask].mean() if neg_mask.sum() > 0 else np.nan
    idi = idi_events - idi_noevents

    return {
        "nri": round(nri, 4) if not np.isnan(nri) else np.nan,
        "nri_events": round(nri_events, 4) if not np.isnan(nri_events) else np.nan,
        "nri_noevents": round(nri_noevents, 4) if not np.isnan(nri_noevents) else np.nan,
        "idi": round(idi, 4) if not np.isnan(idi) else np.nan,
    }

## 6b. Ajuste de Hiperparâmetros (XGB e LGB via RandomizedSearchCV)

In [4]:
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

XGB_PARAM_GRID = {
    "n_estimators": [200, 400, 600],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.7, 0.8, 0.9],
    "colsample_bytree": [0.7, 0.8, 0.9],
    "min_child_weight": [1, 5, 10],
}
LGB_PARAM_GRID = {
    "n_estimators": [200, 400, 600],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.05, 0.1],
    "num_leaves": [31, 63, 127],
    "subsample": [0.7, 0.8, 0.9],
    "colsample_bytree": [0.7, 0.8, 0.9],
}

tscv = TimeSeriesSplit(n_splits=3)

best_xgb_params = {}
best_lgb_params = {}

print("\n" + "=" * 70)
print("AJUSTE DE HIPERPARÂMETROS — XGB e LGB (RandomizedSearchCV, TimeSeriesSplit)")
print("=" * 70)

if HAS_XGB:
    try:
        print("[Tuning] XGBoost...")
        xgb_base = XGBClassifier(
            eval_metric="auc", n_jobs=-1, random_state=42, verbosity=0,
        )
        xgb_search = RandomizedSearchCV(
            xgb_base, XGB_PARAM_GRID,
            n_iter=5, scoring="roc_auc", cv=tscv,
            n_jobs=-1, random_state=42, verbose=0,
        )
        xgb_search.fit(X_train_t, y_train)
        best_xgb_params = xgb_search.best_params_
        print(f"  XGB best AUROC (CV): {xgb_search.best_score_:.4f}")
        print(f"  XGB best params: {best_xgb_params}")
    except Exception as e:
        print(f"  XGB tuning falhou ({e}), usando defaults.")
        best_xgb_params = {}

if HAS_LGB:
    try:
        print("[Tuning] LightGBM...")
        lgb_base = LGBMClassifier(
            n_jobs=-1, random_state=42, verbose=-1,
        )
        lgb_search = RandomizedSearchCV(
            lgb_base, LGB_PARAM_GRID,
            n_iter=5, scoring="roc_auc", cv=tscv,
            n_jobs=-1, random_state=42, verbose=0,
        )
        lgb_search.fit(X_train_t, y_train)
        best_lgb_params = lgb_search.best_params_
        print(f"  LGB best AUROC (CV): {lgb_search.best_score_:.4f}")
        print(f"  LGB best params: {best_lgb_params}")
    except Exception as e:
        print(f"  LGB tuning falhou ({e}), usando defaults.")
        best_lgb_params = {}

## 7. Treinamento dos Modelos (com melhores hiperparâmetros)

In [5]:
USE_SVM = False
if USE_SVM:
    from sklearn.svm import SVC

# ── Define modelos com hiperparâmetros otimizados ─────────────────────────
MODELOS = {
    "M0_Dummy"  : None,
    "M1_LogReg" : LogisticRegression(max_iter=2000, C=0.1, solver="lbfgs"),
    "M1b_Tree"  : DecisionTreeClassifier(max_depth=5, min_samples_leaf=50, random_state=42),
    "M2_RF"     : RandomForestClassifier(n_estimators=300, max_depth=8,
                                          min_samples_leaf=30, n_jobs=-1, random_state=42),
}
if HAS_XGB:
    xgb_final_params = dict(
        eval_metric="auc", n_jobs=-1, random_state=42, verbosity=0,
    )
    xgb_final_params.update(best_xgb_params if best_xgb_params else {
        "n_estimators": 400, "max_depth": 5, "learning_rate": 0.05,
        "subsample": 0.8, "colsample_bytree": 0.8,
    })
    MODELOS["M3_XGB"] = XGBClassifier(**xgb_final_params)

if HAS_LGB:
    lgb_final_params = dict(
        n_jobs=-1, random_state=42, verbose=-1,
    )
    lgb_final_params.update(best_lgb_params if best_lgb_params else {
        "n_estimators": 400, "max_depth": 5, "learning_rate": 0.05,
        "subsample": 0.8, "colsample_bytree": 0.8,
    })
    MODELOS["M4_LGB"] = LGBMClassifier(**lgb_final_params)

if HAS_CAT:
    MODELOS["M5_CAT"] = CatBoostClassifier(
        iterations=400, depth=5, learning_rate=0.05,
        random_seed=42, verbose=0,
        nan_mode="Min",
        cat_features=[i for i, c in enumerate(FEATURES)
                      if not pd.api.types.is_numeric_dtype(df_train[c])],
    )

# Helper: predição de probabilidades
def _predict_proba(nome, clf, X_raw, X_transformed):
    if nome == "M5_CAT" and HAS_CAT:
        return clf.predict_proba(X_raw)[:, 1]
    try:
        return clf.predict_proba(X_transformed)[:, 1]
    except Exception:
        return clf.predict_proba(X_raw)[:, 1]


# ── Treina todos os modelos ────────────────────────────────────────────────
resultados = []
modelos_fit = {}
threshold_youden_por_modelo = {}

for nome, clf in MODELOS.items():
    print(f"\n[{nome}] Treinando...", end=" ")

    if clf is None:
        clf = _Dummy(y_train.mean())
        modelos_fit[nome] = clf
        prob_train = clf.predict_proba(X_train_t)[:, 1]
        prob_valid = clf.predict_proba(X_valid_t)[:, 1]
        prob_test  = clf.predict_proba(X_test_t)[:, 1]
    elif nome == "M5_CAT" and HAS_CAT:
        # CatBoost usa dados brutos com nan_mode="Min" — sem fillna
        clf.fit(X_train, y_train)
        modelos_fit[nome] = clf
        prob_train = clf.predict_proba(X_train)[:, 1]
        prob_valid = clf.predict_proba(X_valid)[:, 1]
        prob_test  = clf.predict_proba(X_test)[:, 1]
    else:
        clf.fit(X_train_t, y_train)
        modelos_fit[nome] = clf
        prob_train = clf.predict_proba(X_train_t)[:, 1]
        prob_valid = clf.predict_proba(X_valid_t)[:, 1]
        prob_test  = clf.predict_proba(X_test_t)[:, 1]

    # Threshold ótimo calculado na validação
    try:
        thr_opt = youden_threshold(y_valid, prob_valid)
    except Exception:
        thr_opt = 0.30
    threshold_youden_por_modelo[nome] = thr_opt

    m_train = avaliar(y_train, prob_train, threshold_opt=thr_opt)
    m_valid = avaliar(y_valid, prob_valid, threshold_opt=thr_opt)
    m_test  = avaliar(y_test,  prob_test,  threshold_opt=thr_opt)

    resultados.append({"modelo": nome, "split": "treino",
                        **{f"trn_{k}": v for k, v in m_train.items()}})
    resultados.append({"modelo": nome, "split": "valid",
                        **{f"vld_{k}": v for k, v in m_valid.items()}})
    resultados.append({"modelo": nome, "split": "teste",
                        **{f"tst_{k}": v for k, v in m_test.items()}})

    print(f"AUROC valid={m_valid.get('auroc','?'):.4f}  teste={m_test.get('auroc','?'):.4f}  "
          f"Threshold_Youden(valid)={thr_opt:.3f}")

## 8. Tabela de Métricas (formato paper)

In [6]:
rows_paper = []
for nome in MODELOS:
    clf = modelos_fit[nome]
    prob_test = _predict_proba(nome, clf, X_test, X_test_t)
    thr_opt = threshold_youden_por_modelo.get(nome, 0.30)
    m = avaliar(y_test, prob_test, threshold_opt=thr_opt)

    rows_paper.append({
        "Modelo"          : nome,
        "AUROC"           : f"{m.get('auroc',''):.4f} [{m.get('auroc_lo',''):.4f}–{m.get('auroc_hi',''):.4f}]",
        "AUPRC"           : f"{m.get('auprc',''):.4f} [{m.get('auprc_lo',''):.4f}–{m.get('auprc_hi',''):.4f}]",
        "Brier"           : f"{m.get('brier',''):.4f}",
        "ECE"             : f"{m.get('ece',''):.4f}",
        "NPV"             : f"{m.get('npv',''):.4f}",
        "MCC"             : f"{m.get('mcc',''):.4f}",
        "Sens@0.30"       : f"{m.get('sens',''):.4f}",
        "Spec@0.30"       : f"{m.get('spec',''):.4f}",
        "PPV@0.30"        : f"{m.get('ppv',''):.4f}",
        "F1@0.30"         : f"{m.get('f1',''):.4f}",
        "Sens@Youden"     : f"{m.get('sens_youden',''):.4f}",
        "Spec@Youden"     : f"{m.get('spec_youden',''):.4f}",
        "F1@Youden"       : f"{m.get('f1_youden',''):.4f}",
        "Threshold_Youden": f"{thr_opt:.4f}",
    })

tab_paper = pd.DataFrame(rows_paper)
print("\nTabela 2 — Desempenho no Conjunto de Teste (IC95% bootstrap)")
print(tab_paper.to_string(index=False))
tab_paper.to_csv(OUT_DIR / "tabela_modelos_teste.csv", index=False)

## 8b. NRI / IDI — Comparação vs Logística (modelo de referência)

In [7]:
print("\n" + "=" * 70)
print("NRI / IDI — Comparação vs M1_LogReg (referência)")
print("=" * 70)

rows_nri_idi = []
if "M1_LogReg" in modelos_fit:
    prob_ref_nri = _predict_proba("M1_LogReg", modelos_fit["M1_LogReg"], X_test, X_test_t)
    for nome in modelos_fit:
        if nome in ("M0_Dummy", "M1_LogReg"):
            continue
        prob_new_nri = _predict_proba(nome, modelos_fit[nome], X_test, X_test_t)
        res = nri_idi(y_test, prob_ref_nri, prob_new_nri)
        rows_nri_idi.append({"modelo": nome, **res})
        print(f"  {nome:20s}  NRI={res['nri']:>7.4f}  IDI={res['idi']:>7.4f}  "
              f"NRI_events={res['nri_events']:>7.4f}  NRI_noevents={res['nri_noevents']:>7.4f}")

pd.DataFrame(rows_nri_idi).to_csv(OUT_DIR / "nri_idi_vs_logreg.csv", index=False)

## Salvar Artefatos

In [8]:
with open(ART_DIR / "modelos_fit.pkl", "wb") as f:
    pickle.dump(modelos_fit, f)

probs_teste = {}
for nome, clf in modelos_fit.items():
    probs_teste[nome] = _predict_proba(nome, clf, X_test, X_test_t)

with open(ART_DIR / "probs_teste.pkl", "wb") as f:
    pickle.dump(probs_teste, f)

with open(ART_DIR / "thresholds_opt.json", "w", encoding="utf-8") as f:
    json.dump(threshold_youden_por_modelo, f, ensure_ascii=False, indent=2)

print(f"\n[03.2] Concluído. Outputs: {OUT_DIR}")